In [2]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# ==========================================
# 1. LOAD DỮ LIỆU THÔ (DATA INGESTION)
# ==========================================
print("--- 1. BẮT ĐẦU ĐỌC DỮ LIỆU ---")
raw_dir = Path("../data/raw")
reg_file = raw_dir / "reg_data.csv"
auth_file = raw_dir / "auth_data.csv"

# Đọc dữ liệu với dấu phân cách ';'
df_reg = pd.read_csv(reg_file, sep=';')
df_auth = pd.read_csv(auth_file, sep=';')

print(f"Bảng Đăng ký (reg_data): {len(df_reg):,} dòng | {df_reg.shape[1]} cột")
print(f"Bảng Đăng nhập (auth_data): {len(df_auth):,} dòng | {df_auth.shape[1]} cột")

# ==========================================
# 2. CHUẨN HÓA THỜI GIAN (TIME NORMALIZATION)
# ==========================================
print("\n--- 2. CHUẨN HÓA THỜI GIAN ---")
df_reg['reg_datetime'] = pd.to_datetime(df_reg['reg_ts'], unit='s')
df_reg['reg_date'] = df_reg['reg_datetime'].dt.normalize()

df_auth['auth_datetime'] = pd.to_datetime(df_auth['auth_ts'], unit='s')
df_auth['auth_date'] = df_auth['auth_datetime'].dt.normalize()

reg_min = df_reg['reg_date'].min().strftime('%Y-%m-%d')
reg_max = df_reg['reg_date'].max().strftime('%Y-%m-%d')
auth_min = df_auth['auth_date'].min().strftime('%Y-%m-%d')
auth_max = df_auth['auth_date'].max().strftime('%Y-%m-%d')

print(f"Khoảng thời gian đăng ký: {reg_min} đến {reg_max}")
print(f"Khoảng thời gian hoạt động: {auth_min} đến {auth_max}")

# ==========================================
# 3. KIỂM ĐỊNH CHẤT LƯỢNG (DATA SANITY AUDIT)
# ==========================================
print("\n--- 3. KIỂM TRA CHẤT LƯỢNG DỮ LIỆU ---")
print("Missing trong reg_data:\n", df_reg.isnull().sum())
print("\nMissing trong auth_data:\n", df_auth.isnull().sum())

exact_dupes = df_auth.duplicated().sum()
print(f"\nSố dòng trùng lặp tuyệt đối (auth_data): {exact_dupes:,}")

# Kiểm tra tính toàn vẹn tham chiếu (Referential Integrity)
reg_uids = set(df_reg['uid'])
auth_uids = set(df_auth['uid'])
ghost_uids = auth_uids - reg_uids
print(f"Số user 'ma' (đăng nhập nhưng không có ngày reg): {len(ghost_uids):,}")

# Kiểm tra nghịch lý thời gian (Time Paradox)
merged_time = pd.merge(df_auth[['uid', 'auth_date']], df_reg[['uid', 'reg_date']], on='uid')
invalid_time = merged_time[merged_time['auth_date'] < merged_time['reg_date']]
print(f"Số lượt đăng nhập TRƯỚC ngày đăng ký: {len(invalid_time):,}")

# ==========================================
# 4. RÚT GỌN VÀ XUẤT DỮ LIỆU SẠCH (EXPORT)
# ==========================================
print("\n--- 4. RÚT GỌN VÀ XUẤT DỮ LIỆU SẠCH ---")
# Giữ 1 bản ghi/user/ngày phục vụ tính DAU & Retention
df_auth_daily = df_auth[['uid', 'auth_date']].drop_duplicates()
print(f"Rút gọn auth_data: từ {len(df_auth):,} -> {len(df_auth_daily):,} dòng active duy nhất.")

# Tạo thư mục processed và xuất file
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

df_reg[['uid', 'reg_date']].to_csv(processed_dir / "cleaned_reg_data.csv", index=False)
df_auth_daily.to_csv(processed_dir / "cleaned_auth_daily.csv", index=False)

print("\n✅ Đã lưu dữ liệu sạch thành công:")
print(" - ../data/processed/cleaned_reg_data.csv")
print(" - ../data/processed/cleaned_auth_daily.csv")

--- 1. BẮT ĐẦU ĐỌC DỮ LIỆU ---
Bảng Đăng ký (reg_data): 1,000,000 dòng | 2 cột
Bảng Đăng nhập (auth_data): 9,601,013 dòng | 2 cột

--- 2. CHUẨN HÓA THỜI GIAN ---
Khoảng thời gian đăng ký: 1998-11-18 đến 2020-09-23
Khoảng thời gian hoạt động: 1998-11-18 đến 2020-09-23

--- 3. KIỂM TRA CHẤT LƯỢNG DỮ LIỆU ---
Missing trong reg_data:
 reg_ts          0
uid             0
reg_datetime    0
reg_date        0
dtype: int64

Missing trong auth_data:
 auth_ts          0
uid              0
auth_datetime    0
auth_date        0
dtype: int64

Số dòng trùng lặp tuyệt đối (auth_data): 0
Số user 'ma' (đăng nhập nhưng không có ngày reg): 0
Số lượt đăng nhập TRƯỚC ngày đăng ký: 0

--- 4. RÚT GỌN VÀ XUẤT DỮ LIỆU SẠCH ---
Rút gọn auth_data: từ 9,601,013 -> 9,601,013 dòng active duy nhất.

✅ Đã lưu dữ liệu sạch thành công:
 - ../data/processed/cleaned_reg_data.csv
 - ../data/processed/cleaned_auth_daily.csv


In [3]:
# Kiểm tra số lượng null trên từng cột của cả 2 bảng
print("--- NULL TRONG BẢNG ĐĂNG KÝ ---")
print(df_reg.isnull().sum())

print("\n--- NULL TRONG BẢNG ĐĂNG NHẬP ---")
print(df_auth.isnull().sum())

--- NULL TRONG BẢNG ĐĂNG KÝ ---
reg_ts          0
uid             0
reg_datetime    0
reg_date        0
dtype: int64

--- NULL TRONG BẢNG ĐĂNG NHẬP ---
auth_ts          0
uid              0
auth_datetime    0
auth_date        0
dtype: int64
